
## **Week 5 Monday: XML parsing (ElementTree), HTML parsing, Data conversion**


**Objective**

- XML basics: what is XML, structure, tree-hierarchy, tags/attributes/text/content

- Using xml.etree.ElementTree in Python: parse files/strings, navigate, find, modify, write

- Data conversion: XML → Python objects / dicts → JSON (or other formats)

- HTML parsing basics: when and why; using BeautifulSoup or html.parser; handling imperfect/malformed HTML

- Quiz + class exercise & best practices summary



### **1.  Recap from Week 4**

- APIs return structured data (often JSON).

- We used requests to fetch and send data.

**Background**

- Many external systems (legacy services, configuration files, RSS feeds, SOAP services) still use XML.

- Some web pages only provide HTML; sometimes you want to extract data (e.g., title, links) via HTML parsing.

- In some projects, you may need to convert data from XML → something easier to store (e.g. dicts, JSON) before inserting to DB or exposing via your own API.

### **2. XML Basics**

**Definition:** Extensible Markup Language (XML) — a markup standard, hierarchical, self-describing.

**Structure:**

```python
<?xml version="1.0" encoding="UTF-8"?>
<library>
  <book id="1" genre="fiction">
    <title>1984</title>
    <author>George Orwell</author>
    <pages>328</pages>
  </book>
  <book id="2" genre="nonfiction">
    <title>Sapiens</title>
    <author>Yuval Noah Harari</author>
    <pages>443</pages>
  </book>
</library>


### Components:

- Elements / tags ```(<book>…</book>)```, nesting

- Attributes ```(id="1", genre="fiction")```

- Text content (between start and end tags)

- Root element, children, siblings

### **3. Using ElementTree**

###   3.1. Importing:


In [3]:
import xml.etree.ElementTree as ET 

### 3.2 Parsing:

**a.) From a file:**

In [4]:
tree = ET.parse('books.xml') ##### (Prof) :\\ 이렇게 작성할 수도 있음 
root = tree.getroot()
print(root)

<Element 'library' at 0x00000213A6D2F450>


**b.) From a string:**

In [5]:
xml_string = '''

<library>
  <book id="1" genre="fiction">
    <title>Introduction to Python</title>
    <author>George Orwell</author>
    <pages>328</pages>
  </book>
  <book id="2" genre="nonfiction">
    <title>Evolution of Human Species</title>
    <author>Yuval Noah Harari</author>
    <pages>443</pages>
  </book>
  <book id="3" genre="nonfiction">
    <title>Introductory Statistics</title>
    <author>David Naval</author>
    <pages>443</pages>
  </book>
</library>
'''

root = ET.fromstring(xml_string)
print(root)

<Element 'library' at 0x00000213A6D2ABD0>


### 3.3 Navigating:


- ```root.tag, root.attrib```

- Loop: ```for book in root.findall('book'):```

    - get sub-elements: ```book.find('title').text, book.get('genre')```

In [ ]:
##### loop 이렇게 되면 pass this student 
for book in root.findall('book'):
    print(book.attrib)

{'id': '1', 'genre': 'fiction'}
{'id': '2', 'genre': 'nonfiction'}
{'id': '3', 'genre': 'nonfiction'}


In [7]:
for book in root.findall('book'):
    print(f"Title: {book.find('title').text} and Genre: {book.get('genre')}")

Title: Introduction to Python and Genre: fiction
Title: Evolution of Human Species and Genre: nonfiction
Title: Introductory Statistics and Genre: nonfiction


### 3.4 Modifying:

- Add or modify attributes: ```book.set('genre', 'dystopian')```

- Change text: ```book.find('pages').text = '330'```

- Remove element: ```root.remove(book)``` or ```book.clear()```

In [8]:
first_book = root.find(".//book[@id='1']")
print(first_book.attrib)

first_book.set('genre','dystopian')
page_count = first_book.find('pages').text

print(f"Pages: {page_count}")




{'id': '1', 'genre': 'fiction'}
Pages: 328


In [9]:
second_book = root.find(".//book[@id='2']")
if second_book:
    # Remove the entire book element from root
    root.remove(second_book)

In [10]:
print("\nModified XML:")
print(ET.tostring(root, encoding='unicode'))


Modified XML:
<library>
  <book id="1" genre="dystopian">
    <title>Introduction to Python</title>
    <author>George Orwell</author>
    <pages>328</pages>
  </book>
  <book id="3" genre="nonfiction">
    <title>Introductory Statistics</title>
    <author>David Naval</author>
    <pages>443</pages>
  </book>
</library>


### 3.5 Writing out:

```python 
tree.write('output.xml', encoding='utf-8', xml_declaration=True)

- XPath (limited support) / findall, find, etc.

- Examples of differences: ```ElementTree``` is in Python standard library. For more advanced needs (namespaces especially), or performance, ```lxml``` is an alternative.

### **4. Data Conversion**

### 4.1 Once you parse an XML, you often want to convert it to a more usable format:

- Convert to Python dicts or objects:


In [ ]:
books = []
##### (Prof) What is a type of this?: Dictionary 
#####  Once you made this, you can access this by 
for book in root.findall('book'):
    
    b = {
    'id': book.get('id'),
    'genre': book.get('genre'),
    'title': book.find('title').text,
    'author': book.find('author').text,
    'pages': int(book.find('pages').text) 
    }
    books.append(b)

books


[{'id': '1',
  'genre': 'dystopian',
  'title': 'Introduction to Python',
  'author': 'George Orwell',
  'pages': 328},
 {'id': '3',
  'genre': 'nonfiction',
  'title': 'Introductory Statistics',
  'author': 'David Naval',
  'pages': 443}]

- Then maybe convert to JSON:

In [12]:
import json

json_string = json.dumps(books, indent=2)
json_string

'[\n  {\n    "id": "1",\n    "genre": "dystopian",\n    "title": "Introduction to Python",\n    "author": "George Orwell",\n    "pages": 328\n  },\n  {\n    "id": "3",\n    "genre": "nonfiction",\n    "title": "Introductory Statistics",\n    "author": "David Naval",\n    "pages": 443\n  }\n]'

### 4.2 Why conversion helps:

- JSON is easier to use with JavaScript / many web APIs

- Databases often expect structured data (dicts or ORM objects)

- Interoperability: your CRUD service might accept JSON, but source is XML/HTML

### **5) HTML Parsing Basics** 

- HTML vs XML:

    - HTML is for web pages; often malformed / less strict than XML

    - XML requires well-formed documents

- Common libraries:

   - Built-in: html.parser (with html.parser.HTMLParser) 

   - Third-party: BeautifulSoup (easy to use, robust), lxml parser, etc. 
 

Example of HTML parsing:

Suppose you have HTML string:

In [13]:
html_string = '''
<html>
  <head><title>My Page</title></head>
  <body>
    <div class="post">
      <h2>First Post</h2>
      <p>This is the post content.</p>
    </div>
    <div class="post">
      <h2>Second Post</h2>
      <p>More content here.</p>
    </div>
  </body>
</html>'''



### Extract titles of all posts:

 -  ```pip install beautifulsoup4```

In [ ]:
##### (Prof) Professor thinks that BeautifulSoup make this html 
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_string, 'html.parser')
posts = soup.find_all('div', class_='post')
##### (Prof) Half class attributes? 
for post in posts:
    title = post.find('h2').text
    print(title)


First Post
Second Post


### **Best Practices**

- Always validate if XML is well-formed (catch parse errors).

- For untrusted XML input, avoid vulnerabilities (XML external entities, etc.).

- Use libraries appropriate for data type: XML parsers are strict; HTML parsers handle malformed HTML better.

- When converting, ensure types are correct (strings, ints, floats) and handle missing data.

- Clean data: strip whitespace, handle missing tags safely (check if tag exists before .text).

- Keep code modular: functions like parse_xml, html_extract, convert_to_json, etc.

### **Quiz** 

- What is the difference between an XML element’s attribute vs text content inside a tag?
# Answer: 

- Which Python method do you use to parse from a string? (parse vs fromstring)
# Answer: fromstring

- When would you use HTML parsing instead of XML parsing?
# Answer: 

- Why convert XML to JSON or dicts in your project?
# Answer:

- HTML must always be well-formed like XML? (True/False).
# Answer: False; structured 

### **Class exercise**

1. Given books.xml with multiple <book> entries. Write code to:

    - Parse the XML

    - Count how many books are in genre = "fiction"

    - Change genre of a specific book (by id) to something else

    - Add a new book entry and write back to output_books.xml


### **Homework**

2. Download HTML of a simple web page of blog posts. Extract titles and links of all posts.

3. Convert extracted data (from XML or HTML) into JSON and print out.